# Platform Inference Entry

Only use this notebook to generate the platform test py file. Select the single code cell below. It defines `predict(X)` for cv2/numpy image input and returns a class label string.


In [ ]:
"""Self-contained platform entry cell for weather image classification.

Use this notebook only for platform py generation. Select this single code cell.
The platform calls predict(X), where X is usually a cv2.imread numpy array.
"""
from pathlib import Path
from typing import Any, Dict
import sys

import numpy as np
import torch
from PIL import Image


def _find_project_root() -> Path:
    starts = []
    if "__file__" in globals():
        starts.append(Path(__file__).resolve().parent)
    starts.append(Path.cwd().resolve())
    names = (".", "weather_competition", "weather_competition-1")
    for start in starts:
        for name in names:
            candidate = (start / name).resolve() if name != "." else start
            if (candidate / "src").exists() and (candidate / "config.py").exists():
                return candidate
    return starts[0]


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config as cfg
from src.dataset import build_transforms
from src.model import build_model
from src.utils import get_device, load_checkpoint, load_checkpoint_payload, load_json

DEFAULT_LABEL = "cloudy"
_MODEL = None
_TRANSFORM = None
_IDX_TO_CLASS = None
_DEVICE = None


def _resolve_path(path_value) -> Path:
    path = Path(path_value)
    if path.exists():
        return path
    candidate = PROJECT_ROOT / path
    if candidate.exists():
        return candidate
    return path


def _load_runtime():
    global _MODEL, _TRANSFORM, _IDX_TO_CLASS, _DEVICE
    if _MODEL is not None:
        return _MODEL, _TRANSFORM, _IDX_TO_CLASS, _DEVICE

    _DEVICE = get_device()
    idx_path = _resolve_path(cfg.IDX_TO_CLASS_PATH)
    model_path = _resolve_path(cfg.BEST_MODEL_PATH)
    _IDX_TO_CLASS = load_json(idx_path)
    num_classes = len(_IDX_TO_CLASS)

    checkpoint = load_checkpoint_payload(model_path, _DEVICE)
    model_name = checkpoint.get("model_name", cfg.MODEL_NAME)
    img_size = int(checkpoint.get("img_size", cfg.IMG_SIZE))
    mean = tuple(checkpoint.get("mean", cfg.MEAN))
    std = tuple(checkpoint.get("std", cfg.STD))

    _MODEL, _ = build_model(model_name, num_classes, pretrained=False, fallback_model_name=cfg.FALLBACK_MODEL_NAME)
    _MODEL = _MODEL.to(_DEVICE)
    load_checkpoint(_MODEL, model_path, _DEVICE)
    _MODEL.eval()
    _TRANSFORM = build_transforms(img_size, is_train=False, mean=mean, std=std)
    return _MODEL, _TRANSFORM, _IDX_TO_CLASS, _DEVICE


def _array_to_pil(array: np.ndarray) -> Image.Image:
    arr = np.asarray(array)
    if arr.ndim == 2:
        return Image.fromarray(arr.astype(np.uint8)).convert("RGB")
    if arr.ndim != 3 or arr.shape[2] not in (1, 3, 4):
        raise ValueError("Unsupported ndarray image shape. Expected HxW, HxWx1, HxWx3, or HxWx4.")
    if arr.dtype != np.uint8:
        arr = np.asarray(arr, dtype=np.float32)
        if arr.size and float(np.nanmax(arr)) <= 1.0:
            arr = arr * 255.0
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    if arr.shape[2] == 1:
        arr = arr[:, :, 0]
        return Image.fromarray(arr).convert("RGB")
    if arr.shape[2] == 4:
        arr = arr[:, :, :3]
    # Platform examples use cv2.imread, so HxWx3 arrays are BGR. Convert BGR to RGB.
    arr = arr[:, :, ::-1]
    return Image.fromarray(arr).convert("RGB")


def _read_image(data: Any) -> Image.Image:
    if isinstance(data, dict):
        data = data.get("image") or data.get("image_path") or data.get("path")
    if isinstance(data, Image.Image):
        return data.convert("RGB")
    if isinstance(data, np.ndarray):
        return _array_to_pil(data)
    if isinstance(data, (str, Path)):
        return Image.open(data).convert("RGB")
    raise ValueError("Unsupported input. Expected cv2 ndarray, image path, PIL image, or dict.")


def handle(data: Any) -> Dict[str, Any]:
    try:
        model, transform, idx_to_class, device = _load_runtime()
        image = _read_image(data)
        tensor = transform(image).unsqueeze(0).to(device)
        with torch.inference_mode():
            logits = model(tensor)
            probs = torch.softmax(logits, dim=1)
            confidence, pred_idx = probs.max(dim=1)
        label = idx_to_class.get(str(int(pred_idx.item())), str(int(pred_idx.item())))
        return {"label": label, "confidence": float(confidence.item())}
    except Exception:
        return {"label": DEFAULT_LABEL, "confidence": 0.0}


def predict(X: Any) -> str:
    """Return one of sunny/cloudy/rainy/snowy for platform scoring."""
    result = handle(X)
    label = result.get("label") if isinstance(result, dict) else None
    if isinstance(label, str) and label:
        return label
    return DEFAULT_LABEL

